# Phase 2: dot detector training and evaluation (Colab)

Full runs of the three Phase 2 detectors on the same data and splits:

| method | target | post-processing |
|---|---|---|
| `resnet_heatmap` (ours) | Gaussian heatmap, focal loss, ImageNet ResNet-34 encoder | peaks → positions + confidence |
| `plain_unet` (original architecture, retrained properly) | disk mask, BCE (pos_weight 5) | connected-component centroids |
| classical | – | local contrast → Otsu → contours |

Thresholds / parameters for all three are selected on **val**; test and test_unseen are only evaluated.

**Runtime → Change runtime type → GPU (T4 or better).** Runs and results are kept on Google Drive, so a disconnected session can be resumed by re-running the cells (training uses `--resume`).

Before the first run, add two Colab secrets (key icon on the left): `GITHUB_TOKEN` (fine-grained, read-only access to the repo) and `GITHUB_REPO` (`username/repo`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/braille-ocr'   # persistent storage for data cache, runs, results
REPO_DIR = '/content/brailleocr'
import os; os.makedirs(DRIVE, exist_ok=True)

In [ ]:
# Clone the private repo without leaving the token in the git config.
from google.colab import userdata
import subprocess
repo = userdata.get('GITHUB_REPO')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{repo}.git", REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://github.com/{repo}.git'], check=True)
else:
    print('repo already cloned; pull manually if needed')
%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
!apt-get -qq install -y liblouis-bin > /dev/null
!pip -q install -e '.[train,dev]'
!python -m pytest -q
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Data

Generation is deterministic given the seed, so this rebuilds the same pages as on the M2 (up to JPEG-encoder differences between OpenCV builds). The generated dataset is cached on Drive as a tarball after the first run.

In [ ]:
CACHE = f'{DRIVE}/synth_seed0.tar'
if os.path.exists(CACHE):
    !mkdir -p data && tar -xf {CACHE} -C data
else:
    !python scripts/fetch_corpus.py
    !python scripts/generate_dataset.py --workers 2
    !tar -cf {CACHE} -C data corpus synth
!cat data/synth/meta.json

## Training

Same steps, batch and crop size for both networks. Each cell can be re-run after a disconnect: `--resume` continues from the last validation checkpoint.

In [ ]:
STEPS = 20000
COMMON = f'--steps {STEPS} --batch 16 --crop 384 --val-every 1000 --val-pages 24 --workers 2 --runs {DRIVE}/runs --resume'
!python scripts/train_detector.py --model resnet_heatmap --name heatmap_r34 {COMMON}

In [ ]:
!python scripts/train_detector.py --model plain_unet --name plainunet {COMMON}

In [ ]:
# Optional ablation: same heatmap model without ImageNet pretraining.
# !python scripts/train_detector.py --model resnet_heatmap --no-pretrained --name heatmap_r34_scratch {COMMON}

## Evaluation on test and test_unseen (all pages, per severity)

In [ ]:
OUT = f'--out {DRIVE}/results/phase2'
!python scripts/evaluate_detector.py --classical --name classical {OUT}
!python scripts/evaluate_detector.py --checkpoint {DRIVE}/runs/plainunet/best.pt --name plainunet {OUT}
!python scripts/evaluate_detector.py --checkpoint {DRIVE}/runs/heatmap_r34/best.pt --name heatmap_r34 {OUT}

In [ ]:
for split in ['test', 'test_unseen']:
    !python scripts/compare_detectors.py classical plainunet heatmap_r34 --split {split} --results {DRIVE}/results/phase2